# LeetCode 777 — Swap Adjacent in LR String bằng Python

Link đề: https://leetcode.com/problems/swap-adjacent-in-lr-string/ · Độ khó: **Medium** · Chủ đề: Two Pointers, String

Skeleton LeetCode:

```python
class Solution:
    def canTransform(self, start: str, end: str) -> bool:
        ...
```

Ghi chú: đề gốc gọi chuỗi đích là `result` (trong phần mô tả), còn tên tham số trong skeleton hiện tại của LeetCode là `end`; hai tên chỉ là một. Ví dụ 3–6 trong mục 2 do notebook tự đặt, đáp án đã kiểm chứng bằng tìm kiếm vét cạn; nếu cần khớp từng chữ, hãy đối chiếu lại đề gốc.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: chuỗi chỉ gồm `L`, `R`, `X`; một nước đi là thay một `XL` bằng `LX` (chữ `L` nhích **sang trái** qua một `X`) hoặc một `RX` bằng `XR` (chữ `R` nhích **sang phải** qua một `X`); hỏi có thể biến `start` thành `end` bằng một dãy nước đi hay không.
2. Nắm các công cụ Python: duyệt chuỗi bằng chỉ số, hai con trỏ, `str.replace`, `enumerate`, `zip`, `collections.deque` cho BFS trên tập trạng thái, cắt chuỗi `s[:i] + t + s[i+2:]`.
3. Đi từ **tìm kiếm vét cạn (BFS mọi chuỗi đạt được)** → **hai con trỏ trên các ký tự khác `X`** → **danh sách (ký tự, vị trí) rồi so từng cặp** → **đếm tiền tố** → **mô phỏng tham lam dựng ra dãy nước đi**.
4. Hiểu mấu chốt (bất biến): `L` chỉ đi sang trái, `R` chỉ đi sang phải, và **không bao giờ vượt qua nhau**. Do đó nếu bỏ hết `X` thì hai chuỗi phải **y hệt nhau**; và ký tự thứ `k` (khác `X`) ở `start` so với ở `end`: `L` không được nằm lệch về **bên trái** của vị trí đích, `R` không được lệch về **bên phải**.
5. Tránh các lỗi kinh điển: chỉ so sánh số lượng `L`, `R`, `X` (bỏ qua thứ tự), quên điều kiện hướng đi, so sánh sai chiều (`<` với `>`), lặp con trỏ không bỏ qua `X`, và cho `L`/`R` đi ngược chiều.

## 2. Đề bài & ví dụ

Trong một chuỗi gồm các ký tự `'L'`, `'R'` và `'X'`, ví dụ `"RXXLRXRXL"`, một **nước đi** là:

- thay một lần xuất hiện của `"XL"` bằng `"LX"`, hoặc
- thay một lần xuất hiện của `"RX"` bằng `"XR"`.

Cho chuỗi bắt đầu `start` và chuỗi đích `end` (gọi là `result` trong mô tả gốc), trả về `True` **khi và chỉ khi** tồn tại một dãy nước đi biến `start` thành `end`.

**Ví dụ 1**

- Input: `start = "RXXLRXRXL"`, `end = "XRLXXRRLX"`
- Output: `True`
- Giải thích: `RXXLRXRXL → XRXLRXRXL → XRLXRXRXL → XRLXXRRXL → XRLXXRRLX`.

**Ví dụ 2**

- Input: `start = "X"`, `end = "L"`
- Output: `False`

**Ví dụ 3 (tự đặt, đối chiếu lại đề gốc nếu cần)**

- Input: `start = "XL"`, `end = "LX"`
- Output: `True` (đúng một nước `XL → LX`).

**Ví dụ 4 (tự đặt)**

- Input: `start = "LX"`, `end = "XL"`
- Output: `False` (`L` không đi được sang phải).

**Ví dụ 5 (tự đặt)**

- Input: `start = "RL"`, `end = "LR"`
- Output: `False` (`R` và `L` không thể đổi chỗ nhau; không có `X` để di chuyển).

**Ví dụ 6 (tự đặt)**

- Input: `start = "RXXL"`, `end = "XRLX"`
- Output: `True` (`R` sang phải một bước, `L` sang trái một bước).

**Ràng buộc:**

- `1 <= start.length <= 10^4`
- `start.length == end.length`
- `start` và `end` chỉ gồm các ký tự `'L'`, `'R'` và `'X'`.

In [1]:
from typing import List
from collections import deque
import random
import time

examples = [
    ("RXXLRXRXL", "XRLXXRRLX", True),
    ("X", "L", False),
    ("XL", "LX", True),
    ("LX", "XL", False),
    ("RL", "LR", False),
    ("RXXL", "XRLX", True),
]
for start, end, expected in examples:
    print(f"start={start!r}, end={end!r} -> kỳ vọng {expected}")

start='RXXLRXRXL', end='XRLXXRRLX' -> kỳ vọng True
start='X', end='L' -> kỳ vọng False
start='XL', end='LX' -> kỳ vọng True
start='LX', end='XL' -> kỳ vọng False
start='RL', end='LR' -> kỳ vọng False
start='RXXL', end='XRLX' -> kỳ vọng True


## 3. Kiến thức nền

### 3.1. Chuỗi, chỉ số và thay thế một đoạn

Chuỗi trong Python **không sửa tại chỗ được** (immutable): muốn "đổi" ta tạo chuỗi mới bằng cách ghép: `s[:i] + "LX" + s[i+2:]` thay hai ký tự ở vị trí `i, i+1`. `s[i:i+2]` lấy đúng hai ký tự đó để kiểm tra có phải `"XL"` hay `"RX"` không.

In [2]:
s = "RXXLRXRXL"
i = 3 - 1                                  # "XL" nằm ở vị trí 2..3
print(s[i:i + 2])
if s[i:i + 2] == "XL":
    t = s[:i] + "LX" + s[i + 2:]
    print(s, "->", t)
print("tất cả vị trí có nước đi:", [(i, s[i:i + 2]) for i in range(len(s) - 1) if s[i:i + 2] in ("XL", "RX")])

XL
RXXLRXRXL -> RXLXRXRXL
tất cả vị trí có nước đi: [(0, 'RX'), (2, 'XL'), (4, 'RX'), (6, 'RX'), (7, 'XL')]


### 3.2. `str.replace` và "bỏ hết `X`"

`s.replace("X", "")` xoá mọi ký tự `X`, cho ta **dãy các `L`/`R` theo thứ tự xuất hiện**. Vì `L` và `R` không vượt qua nhau (ta sẽ thấy ở mục 4), dãy này là **bất biến** qua các nước đi.

In [3]:
a, b = "RXXLRXRXL", "XRLXXRRLX"
print(a.replace("X", ""), b.replace("X", ""), a.replace("X", "") == b.replace("X", ""))
print("XL".replace("X", ""), "LX".replace("X", ""))

RLRRL RLRRL True
L L


### 3.3. `enumerate`, `zip` và danh sách các cặp (ký tự, vị trí)

`enumerate(s)` cho `(chỉ số, ký tự)`; `zip(a, b)` ghép từng phần tử hai dãy song song (dừng ở dãy ngắn hơn). Ta dùng chúng để lấy **vị trí của các ký tự khác `X`** và so sánh từng cặp "ký tự thứ `k` ở `start` với ký tự thứ `k` ở `end`".

In [4]:
s = "RXXLRXRXL"
pos = [(c, i) for i, c in enumerate(s) if c != "X"]
print(pos)
print(list(zip([1, 2, 3], "ab")))

[('R', 0), ('L', 3), ('R', 4), ('R', 6), ('L', 8)]
[(1, 'a'), (2, 'b')]


### 3.4. Hai con trỏ và việc "bỏ qua" ký tự không quan trọng

Kỹ thuật **hai con trỏ**: mỗi chuỗi một con trỏ `i`, `j`; mỗi lần cho cả hai **nhảy qua các `X`** để dừng ở ký tự `L`/`R` kế tiếp; so sánh hai ký tự đó rồi cùng tiến. Cách này không cần tạo chuỗi hay danh sách mới.

In [5]:
def skip_x(s, i):
    while i < len(s) and s[i] == "X":
        i += 1
    return i

s = "XXRXL"
i = skip_x(s, 0)
print(i, s[i])
i = skip_x(s, i + 1)
print(i, s[i])
print(skip_x(s, i + 1), "= len(s) nghĩa là đã hết ký tự")

2 R
4 L
5 = len(s) nghĩa là đã hết ký tự


### 3.5. BFS trên tập trạng thái (để vét cạn)

Bài toán "có biến đổi được không" có thể giải bằng **tìm kiếm trên đồ thị trạng thái**: mỗi chuỗi là một đỉnh, mỗi nước đi là một cạnh. **BFS** với `deque` và tập `seen` liệt kê **mọi chuỗi đạt được** từ `start`. Chỉ khả thi với chuỗi ngắn (số chuỗi có thể tăng như hàm mũ), nên ta dùng nó làm **baseline và oracle**.

In [6]:
q = deque(["a"])
seen = {"a"}
while q:
    cur = q.popleft()
    for nxt in (cur + "b", cur + "c"):
        if len(nxt) <= 3 and nxt not in seen:
            seen.add(nxt)
            q.append(nxt)
print(len(seen), sorted(seen)[:6], "...")

7 ['a', 'ab', 'abb', 'abc', 'ac', 'acb'] ...


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Hãy xem các chữ như **xe trên đường ray một chiều**:

- `R` là xe chỉ **chạy sang phải**;
- `L` là xe chỉ **chạy sang trái**;
- `X` là **ô trống**. Xe chỉ nhích một bước vào **ô trống kề bên**, và **không thể đi xuyên qua** xe khác.

Từ đó rút ra:

1. Thứ tự các xe (bỏ ô trống) **không bao giờ đổi** — hai xe không vượt qua nhau. Ví dụ `RL → LR` bất khả vì `R` và `L` cần đổi chỗ.
2. Một xe `R` ở vị trí `a` chỉ có thể kết thúc ở vị trí `b ≥ a`; xe `L` ở `a` chỉ có thể kết thúc ở `b ≤ a`.

Kiểm tra bằng tay ví dụ 1: `start = RXXLRXRXL`, `end = XRLXXRRLX`.

| xe thứ | loại | vị trí ở `start` | vị trí ở `end` | hợp lệ? |
|---|---|---|---|---|
| 1 | R | 0 | 1 | `0 ≤ 1` ✔ |
| 2 | L | 3 | 2 | `2 ≤ 3` ✔ |
| 3 | R | 4 | 5 | `4 ≤ 5` ✔ |
| 4 | R | 6 | 6 | `6 ≤ 6` ✔ |
| 5 | L | 8 | 7 | `7 ≤ 8` ✔ |

Cùng dãy xe `R L R R L` và mọi xe đi đúng chiều → `True`. Ví dụ 2: `X` vs `L` — dãy xe khác nhau (rỗng vs `L`) → `False`.

In [7]:
def positions(s):
    return [(c, i) for i, c in enumerate(s) if c != "X"]

start, end = "RXXLRXRXL", "XRLXXRRLX"
for (c1, i), (c2, j) in zip(positions(start), positions(end)):
    ok = (c1 == c2) and (i <= j if c1 == "R" else i >= j)
    print(f"{c1}->{c2}: {i} -> {j}  {'OK' if ok else 'SAI'}")

R->R: 0 -> 1  OK
L->L: 3 -> 2  OK
R->R: 4 -> 5  OK
R->R: 6 -> 6  OK
L->L: 8 -> 7  OK


## 5. Cách 1 — Brute Force / baseline: BFS mọi chuỗi đạt được

### 5.1. Ý tưởng

Dựng **tập mọi chuỗi** có thể đạt được từ `start` bằng BFS: với mỗi chuỗi, thử mọi vị trí `i`; nếu `s[i:i+2] == "XL"` ta sinh `s[:i] + "LX" + s[i+2:]`; nếu `== "RX"` sinh `s[:i] + "XR" + s[i+2:]`. Trả lời: `end` có thuộc tập đó không.

In [8]:
def reachable(start):
    seen = {start}
    queue = deque([start])
    while queue:
        s = queue.popleft()
        for i in range(len(s) - 1):
            pair = s[i:i + 2]
            if pair == "XL":
                t = s[:i] + "LX" + s[i + 2:]
            elif pair == "RX":
                t = s[:i] + "XR" + s[i + 2:]
            else:
                continue
            if t not in seen:
                seen.add(t)
                queue.append(t)
    return seen

def can_transform_bfs(start, end):
    return end in reachable(start)

for start, end, expected in examples:
    got = can_transform_bfs(start, end)
    print(f"{start!r} -> {end!r}: {got} (số chuỗi đạt được từ start: {len(reachable(start))})")
    assert got == expected

'RXXLRXRXL' -> 'XRLXXRRLX': True (số chuỗi đạt được từ start: 42)
'X' -> 'L': False (số chuỗi đạt được từ start: 1)
'XL' -> 'LX': True (số chuỗi đạt được từ start: 2)
'LX' -> 'XL': False (số chuỗi đạt được từ start: 1)
'RL' -> 'LR': False (số chuỗi đạt được từ start: 1)
'RXXL' -> 'XRLX': True (số chuỗi đạt được từ start: 6)


### 5.2. Độ phức tạp và nút thắt

Số chuỗi đạt được có thể lên tới hàm mũ theo độ dài (mỗi vị trí `X`/`L`/`R` có thể phân bố khác nhau), mỗi chuỗi tốn `O(n)` để sinh các chuỗi kề. Với `n = 10^4` hoàn toàn không khả thi. Hãy đo số trạng thái:

In [9]:
for n in (6, 8, 10, 12):
    s = "R" + "X" * (n - 2) + "L"
    t0 = time.perf_counter()
    cnt = len(reachable(s))
    print(f"n={n:2d} {s}: {cnt:6d} chuỗi đạt được, {1000 * (time.perf_counter() - t0):8.2f} ms")
for n in (6, 8, 10):
    s = "RX" * (n // 2)
    print(f"n={n:2d} {s}: {len(reachable(s)):6d} chuỗi đạt được")

n= 6 RXXXXL:     15 chuỗi đạt được,     0.04 ms
n= 8 RXXXXXXL:     28 chuỗi đạt được,     0.06 ms
n=10 RXXXXXXXXL:     45 chuỗi đạt được,     0.14 ms
n=12 RXXXXXXXXXXL:     66 chuỗi đạt được,     0.16 ms
n= 6 RXRXRX:     14 chuỗi đạt được
n= 8 RXRXRXRX:     42 chuỗi đạt được
n=10 RXRXRXRXRX:    132 chuỗi đạt được


Nút thắt: ta **liệt kê cả những chuỗi trung gian không liên quan** trong khi câu hỏi chỉ cần biết `end` có đạt được hay không. Bất biến ở mục 4 cho phép **kiểm tra trực tiếp**.

## 6. Ý tưởng thuật toán / bước đột phá

**Bất biến 1 (thứ tự).** Mỗi nước đi chỉ đổi chỗ một ký tự `L`/`R` với một `X` kề nó, không bao giờ đổi chỗ hai ký tự `L`/`R` với nhau. Vậy dãy các ký tự `L`/`R` (bỏ `X`) **không đổi**: `start.replace("X", "") == end.replace("X", "")` là điều kiện **cần**.

**Bất biến 2 (hướng).** `R` chỉ dời sang phải, `L` chỉ dời sang trái. Nếu `a_k` và `b_k` lần lượt là vị trí ký tự thứ `k` (khác `X`) trong `start` và `end` thì: nếu nó là `R` cần `a_k ≤ b_k`; nếu là `L` cần `a_k ≥ b_k`. Đây cũng là điều kiện **cần**.

**Đủ.** Hai điều kiện cần trên cũng **đủ**: ta có thể thực hiện các nước đi để đưa mọi ký tự về đúng vị trí đích (chứng minh ở mục 10). Vậy **thuật toán kiểm tra** gồm: so dãy `L`/`R` rồi so từng cặp vị trí — thời gian `O(n)`, không cần mô phỏng.

Hai con trỏ làm gọn hai kiểm tra thành một lượt duyệt: `i` chạy trên `start`, `j` trên `end`, mỗi lần bỏ qua các `X` rồi so ký tự và vị trí.

## 7. Các cách tiếp theo

Tổng cộng có 5 cách: Cách 1 (BFS vét cạn, mục 5) và bốn cách dưới đây. Mỗi cách: giải thích từng đoạn → code demo → code nộp LeetCode → độ phức tạp.

### 7.1. Cách 2 — Hai con trỏ trên các ký tự khác `X` (`O(n)`, `O(1)` bộ nhớ)

Giải thích từng đoạn:

1. `i`, `j` cùng bắt đầu từ `0`.
2. Mỗi vòng: đẩy `i` qua các `X` của `start`, đẩy `j` qua các `X` của `end`.
3. Nếu cả hai đã hết chuỗi (`i >= n` và `j >= n`) → `True` (đã khớp mọi xe).
4. Nếu chỉ một bên hết, hoặc `start[i] != end[j]` → `False` (dãy xe khác nhau).
5. Nếu xe là `L` mà `i < j` (xe `L` ở `start` nằm **bên trái** vị trí đích → phải đi sang phải) → `False`. Nếu xe là `R` mà `i > j` → `False`.
6. Tiến `i += 1`, `j += 1`.

In [10]:
def two_pointers_demo(start, end):
    n = len(start)
    i = j = 0
    while True:
        while i < n and start[i] == "X":
            i += 1
        while j < n and end[j] == "X":
            j += 1
        if i >= n and j >= n:
            print("  hết cả hai -> True")
            return True
        if i >= n or j >= n:
            print(f"  một bên hết (i={i}, j={j}) -> False")
            return False
        print(f"  xe: start[{i}]={start[i]} vs end[{j}]={end[j]}")
        if start[i] != end[j]:
            print("  khác loại -> False")
            return False
        if start[i] == "L" and i < j:
            print("  L phải đi sang phải -> False")
            return False
        if start[i] == "R" and i > j:
            print("  R phải đi sang trái -> False")
            return False
        i, j = i + 1, j + 1

print(two_pointers_demo("RXXLRXRXL", "XRLXXRRLX"))
print(two_pointers_demo("LX", "XL"))

  xe: start[0]=R vs end[1]=R
  xe: start[3]=L vs end[2]=L
  xe: start[4]=R vs end[5]=R
  xe: start[6]=R vs end[6]=R
  xe: start[8]=L vs end[7]=L
  hết cả hai -> True
True
  xe: start[0]=L vs end[1]=L
  L phải đi sang phải -> False
False


In [11]:
class Sol2:
    """Cách 2: hai con trỏ."""
    def canTransform(self, start: str, end: str) -> bool:
        n = len(start)
        i = j = 0
        while True:
            while i < n and start[i] == "X":
                i += 1
            while j < n and end[j] == "X":
                j += 1
            if i >= n and j >= n:
                return True
            if i >= n or j >= n or start[i] != end[j]:
                return False
            if start[i] == "L" and i < j:
                return False
            if start[i] == "R" and i > j:
                return False
            i, j = i + 1, j + 1

for start, end, expected in examples:
    assert Sol2().canTransform(start, end) == expected
print("Sol2 OK")

Sol2 OK


**Độ phức tạp:** mỗi con trỏ đi qua chuỗi tối đa một lần: Time `O(n)`, Space `O(1)`.

### 7.2. Cách 3 — Danh sách (ký tự, vị trí) rồi so từng cặp

Cùng ý tưởng nhưng viết "chia để dễ đọc":

1. Lấy `a = [(c, i) ...]` từ `start` và `b = [(c, j) ...]` từ `end` chỉ với `c != "X"`.
2. Nếu độ dài khác nhau → `False`.
3. Với từng cặp `((c1, i), (c2, j))`: `c1 == c2`; nếu `c1 == "R"` cần `i <= j`; nếu `c1 == "L"` cần `i >= j`.

In [12]:
def pairs_demo(start, end):
    a = [(c, i) for i, c in enumerate(start) if c != "X"]
    b = [(c, j) for j, c in enumerate(end) if c != "X"]
    print("  a =", a)
    print("  b =", b)
    if len(a) != len(b):
        return False
    for (c1, i), (c2, j) in zip(a, b):
        ok = c1 == c2 and (i <= j if c1 == "R" else i >= j)
        print(f"  {c1}{i} vs {c2}{j}: {'ok' if ok else 'SAI'}")
        if not ok:
            return False
    return True

print(pairs_demo("RXXLRXRXL", "XRLXXRRLX"))
print(pairs_demo("RXL", "XLR"))

  a = [('R', 0), ('L', 3), ('R', 4), ('R', 6), ('L', 8)]
  b = [('R', 1), ('L', 2), ('R', 5), ('R', 6), ('L', 7)]
  R0 vs R1: ok
  L3 vs L2: ok
  R4 vs R5: ok
  R6 vs R6: ok
  L8 vs L7: ok
True
  a = [('R', 0), ('L', 2)]
  b = [('L', 1), ('R', 2)]
  R0 vs L1: SAI
False


In [13]:
class Sol3:
    """Cách 3: danh sách (ký tự, vị trí) và so từng cặp."""
    def canTransform(self, start: str, end: str) -> bool:
        a = [(c, i) for i, c in enumerate(start) if c != "X"]
        b = [(c, j) for j, c in enumerate(end) if c != "X"]
        if len(a) != len(b):
            return False
        for (c1, i), (c2, j) in zip(a, b):
            if c1 != c2:
                return False
            if c1 == "R" and i > j:
                return False
            if c1 == "L" and i < j:
                return False
        return True

for start, end, expected in examples:
    assert Sol3().canTransform(start, end) == expected
print("Sol3 OK")

Sol3 OK


**Độ phức tạp:** Time `O(n)`, Space `O(n)` cho hai danh sách. Dễ viết đúng hơn hai con trỏ vì không có vòng `while` lồng nhau.

### 7.3. Cách 4 — Đếm tiền tố

Điều kiện "`R` không lệch phải, `L` không lệch trái" có thể phát biểu qua **số lượng trong mỗi tiền tố**. Với mỗi độ dài tiền tố `p`:

- số `R` trong `start[:p]` **≥** số `R` trong `end[:p]` (các `R` ở `end` đến **muộn hơn** hoặc bằng);
- số `L` trong `start[:p]` **≤** số `L` trong `end[:p]` (các `L` ở `end` đến **sớm hơn** hoặc bằng).

Kết hợp với điều kiện "bỏ `X` thì hai chuỗi giống nhau" (để thứ tự không đổi) ta được một kiểm tra thay thế hoàn toàn. (Chỉ đếm tiền tố mà bỏ điều kiện thứ tự là **sai**: `RL → LR` thoả các bất đẳng thức đếm nhưng không thể thực hiện.)

In [14]:
def prefix_demo(start, end):
    if start.replace("X", "") != end.replace("X", ""):
        print("  dãy xe khác nhau -> False")
        return False
    rs = ls = re_ = le = 0
    for p, (a, b) in enumerate(zip(start, end), 1):
        rs += a == "R"; ls += a == "L"
        re_ += b == "R"; le += b == "L"
        bad = rs < re_ or ls > le
        print(f"  p={p}: R(start)={rs} R(end)={re_} | L(start)={ls} L(end)={le} {'<- vi phạm' if bad else ''}")
        if bad:
            return False
    return True

print(prefix_demo("RXXL", "XRLX"))
print(prefix_demo("LX", "XL"))

  p=1: R(start)=1 R(end)=0 | L(start)=0 L(end)=0 
  p=2: R(start)=1 R(end)=1 | L(start)=0 L(end)=0 
  p=3: R(start)=1 R(end)=1 | L(start)=0 L(end)=1 
  p=4: R(start)=1 R(end)=1 | L(start)=1 L(end)=1 
True
  p=1: R(start)=0 R(end)=0 | L(start)=1 L(end)=0 <- vi phạm
False


In [15]:
class Sol4:
    """Cách 4: so dãy xe rồi đếm tiền tố."""
    def canTransform(self, start: str, end: str) -> bool:
        if start.replace("X", "") != end.replace("X", ""):
            return False
        r_start = l_start = r_end = l_end = 0
        for a, b in zip(start, end):
            if a == "R":
                r_start += 1
            elif a == "L":
                l_start += 1
            if b == "R":
                r_end += 1
            elif b == "L":
                l_end += 1
            if r_start < r_end or l_start > l_end:
                return False
        return True

for start, end, expected in examples:
    assert Sol4().canTransform(start, end) == expected
print("Sol4 OK")

Sol4 OK


**Độ phức tạp:** Time `O(n)`, Space `O(n)` cho `replace` (có thể thay bằng so sánh theo chỉ số để còn `O(1)`).

### 7.4. Cách 5 — Mô phỏng tham lam: dựng ra dãy nước đi

Ý tưởng "xây dựng": nếu dãy xe hai chuỗi giống nhau, ta **ghép ký tự thứ `k` ở `start` với ký tự thứ `k` ở `end`** (vị trí hiện tại `p_k`, vị trí đích `b_k`) rồi **thực sự di chuyển** từng xe:

1. Duyệt các xe `R` **từ phải sang trái**: nhích xe sang phải khi `p_k < b_k` và ô kề bên phải còn trống.
2. Duyệt các xe `L` **từ trái sang phải**: nhích xe sang trái khi `p_k > b_k` và ô kề bên trái còn trống.
3. Cuối cùng so sánh vị trí hiện tại với vị trí đích: bằng nhau ⇒ `True`, kèm **danh sách nước đi** làm bằng chứng.

Cách này chạy tốn `O(số nước đi)` — có thể tới `O(n²)` — không dùng để nộp, nhưng cho ta **lời giải tường minh** và là cơ sở của chứng minh tính đủ ở mục 10.

In [16]:
def construct_moves(start, end):
    """Trả về (đạt được?, danh sách nước đi (vị trí ký tự trước nước, ký tự))."""
    if start.replace("X", "") != end.replace("X", ""):
        return False, []
    chars = [c for c in start if c != "X"]
    cur = [i for i, c in enumerate(start) if c != "X"]
    goal = [j for j, c in enumerate(end) if c != "X"]
    m = len(chars)
    moves = []
    for k in range(m - 1, -1, -1):                      # R từ phải sang trái
        if chars[k] != "R":
            continue
        while cur[k] < goal[k] and (k == m - 1 or cur[k + 1] > cur[k] + 1):
            moves.append((cur[k], "R"))
            cur[k] += 1
    for k in range(m):                                  # L từ trái sang phải
        if chars[k] != "L":
            continue
        while cur[k] > goal[k] and (k == 0 or cur[k - 1] < cur[k] - 1):
            moves.append((cur[k], "L"))
            cur[k] -= 1
    return cur == goal, moves

def apply_moves(start, moves):
    s = list(start)
    for pos, c in moves:
        if c == "R":
            assert s[pos] == "R" and s[pos + 1] == "X", "nước đi R không hợp lệ"
            s[pos], s[pos + 1] = "X", "R"
        else:
            assert s[pos] == "L" and s[pos - 1] == "X", "nước đi L không hợp lệ"
            s[pos], s[pos - 1] = "X", "L"
    return "".join(s)

ok, moves = construct_moves("RXXLRXRXL", "XRLXXRRLX")
print(ok, len(moves), "nước:", moves)
print("áp dụng toàn bộ:", apply_moves("RXXLRXRXL", moves))

True 4 nước: [(4, 'R'), (0, 'R'), (3, 'L'), (8, 'L')]
áp dụng toàn bộ: XRLXXRRLX


In [17]:
class Sol5:
    """Cách 5: mô phỏng tham lam, quyết định bằng cách thực sự di chuyển các xe."""
    def canTransform(self, start: str, end: str) -> bool:
        if start.replace("X", "") != end.replace("X", ""):
            return False
        chars = [c for c in start if c != "X"]
        cur = [i for i, c in enumerate(start) if c != "X"]
        goal = [j for j, c in enumerate(end) if c != "X"]
        m = len(chars)
        for k in range(m - 1, -1, -1):
            if chars[k] == "R":
                while cur[k] < goal[k] and (k == m - 1 or cur[k + 1] > cur[k] + 1):
                    cur[k] += 1
        for k in range(m):
            if chars[k] == "L":
                while cur[k] > goal[k] and (k == 0 or cur[k - 1] < cur[k] - 1):
                    cur[k] -= 1
        return cur == goal

for start, end, expected in examples:
    assert Sol5().canTransform(start, end) == expected
print("Sol5 OK")

Sol5 OK


**Độ phức tạp:** Time `O(n + D)` với `D = Σ|a_k − b_k|` là tổng quãng đường (tới `O(n²)` trong tệ nhất), Space `O(n)`. Chậm hơn các cách khác nhưng **tự chứng minh** vì nó thực sự thực hiện các nước đi hợp lệ.

## 8. TRACE từng bước

### 8.1. Trace hai con trỏ (Cách 2)

Với `start = "RXXLRXRXL"`, `end = "XRLXXRRLX"`:

| bước | `i` (sau bỏ `X`) | `j` (sau bỏ `X`) | `start[i]` | `end[j]` | kiểm tra | hành động |
|---|---|---|---|---|---|---|
| 1 | 0 | 1 | R | R | `R`, `i=0 ≤ j=1` ✔ | `i=1, j=2` |
| 2 | 3 | 2 | L | L | `L`, `i=3 ≥ j=2` ✔ | `i=4, j=3` |
| 3 | 4 | 5 | R | R | `4 ≤ 5` ✔ | `i=5, j=6` |
| 4 | 6 | 6 | R | R | `6 ≤ 6` ✔ | `i=7, j=7` |
| 5 | 8 | 7 | L | L | `8 ≥ 7` ✔ | `i=9, j=8` |
| 6 | 9 (hết) | 9 (hết; `j=8` bỏ `X` thành 9) | — | — | cả hai hết | `True` |

Hàm `trace_two_pointers` bên dưới in đúng bảng này:

In [18]:
def trace_two_pointers(start, end):
    n = len(start)
    i = j = 0
    step = 0
    print(f"start={start}  end={end}")
    while True:
        while i < n and start[i] == "X":
            i += 1
        while j < n and end[j] == "X":
            j += 1
        step += 1
        if i >= n and j >= n:
            print(f"bước {step}: i={i}, j={j}: cả hai hết -> True")
            return True
        if i >= n or j >= n:
            print(f"bước {step}: i={i}, j={j}: một bên hết -> False")
            return False
        a, b = start[i], end[j]
        verdict = "ok"
        if a != b:
            verdict = "khác loại"
        elif a == "L" and i < j:
            verdict = "L bị lệch trái (cần đi phải)"
        elif a == "R" and i > j:
            verdict = "R bị lệch phải (cần đi trái)"
        print(f"bước {step}: i={i} j={j} start[i]={a} end[j]={b} -> {verdict}")
        if verdict != "ok":
            return False
        i, j = i + 1, j + 1

trace_two_pointers("RXXLRXRXL", "XRLXXRRLX")
print()
trace_two_pointers("XXRXL", "XRXXL")
print()
trace_two_pointers("RXXL", "XLXR")

start=RXXLRXRXL  end=XRLXXRRLX
bước 1: i=0 j=1 start[i]=R end[j]=R -> ok
bước 2: i=3 j=2 start[i]=L end[j]=L -> ok
bước 3: i=4 j=5 start[i]=R end[j]=R -> ok
bước 4: i=6 j=6 start[i]=R end[j]=R -> ok
bước 5: i=8 j=7 start[i]=L end[j]=L -> ok
bước 6: i=9, j=9: cả hai hết -> True

start=XXRXL  end=XRXXL
bước 1: i=2 j=1 start[i]=R end[j]=R -> R bị lệch phải (cần đi trái)

start=RXXL  end=XLXR
bước 1: i=0 j=1 start[i]=R end[j]=L -> khác loại


False

Quan sát: ví dụ thứ hai (`XXRXL → XRXXL`) fail ở bước 1: `R` ở vị trí 2 nhưng đích ở vị trí 1 — phải đi **trái**. Ví dụ thứ ba fail vì dãy xe `RL` khác `LR`.

### 8.2. Trace Cách 3: bảng so sánh từng cặp

`pairs_demo` ở mục 7.2 in danh sách `a`, `b` và từng cặp. Thử một trường hợp có một cặp vi phạm giữa chừng:

In [19]:
print(pairs_demo("XRXXLRX", "RXXXXLR"))

  a = [('R', 1), ('L', 4), ('R', 5)]
  b = [('R', 0), ('L', 5), ('R', 6)]
  R1 vs R0: SAI
False


Ở đây `R(1) → R(0)` đã vi phạm (R lùi trái), dừng ngay. Cách 3 dừng ở **cặp đầu tiên vi phạm**.

### 8.3. Trace Cách 4: bảng đếm tiền tố

Với `start = "RXXL"`, `end = "XRLX"`: mỗi dòng cho thấy đếm `R`, `L` ở hai chuỗi; điều kiện `R(start) ≥ R(end)` và `L(start) ≤ L(end)` phải đúng ở **mọi** tiền tố.

In [20]:
prefix_demo("RXXL", "XRLX")
print()
prefix_demo("XRXL", "RXLX")

  p=1: R(start)=1 R(end)=0 | L(start)=0 L(end)=0 
  p=2: R(start)=1 R(end)=1 | L(start)=0 L(end)=0 
  p=3: R(start)=1 R(end)=1 | L(start)=0 L(end)=1 
  p=4: R(start)=1 R(end)=1 | L(start)=1 L(end)=1 

  p=1: R(start)=0 R(end)=1 | L(start)=0 L(end)=0 <- vi phạm


False

Trường hợp thứ hai vi phạm ở `p = 1`: `R(end) = 1 > R(start) = 0` — tức là `R` ở `end` đến sớm hơn ở `start`, nghĩa là phải lùi trái.

### 8.4. Trace Cách 5: các xe di chuyển từng bước

In vị trí hiện tại của từng xe sau mỗi nước đi:

In [21]:
def trace_moves(start, end):
    ok, moves = construct_moves(start, end)
    print(f"start={start} end={end} -> {ok}, {len(moves)} nước")
    s = start
    print(f"   {s}")
    for pos, c in moves:
        s = apply_moves(s, [(pos, c)])
        print(f"   {s}   ({c} ở {pos} -> {pos + 1 if c == 'R' else pos - 1})")
    assert (s == end) == ok
    return ok

trace_moves("RXXLRXRXL", "XRLXXRRLX")
trace_moves("RXXR", "XXRR")

start=RXXLRXRXL end=XRLXXRRLX -> True, 4 nước
   RXXLRXRXL
   RXXLXRRXL   (R ở 4 -> 5)
   XRXLXRRXL   (R ở 0 -> 1)
   XRLXXRRXL   (L ở 3 -> 2)
   XRLXXRRLX   (L ở 8 -> 7)
start=RXXR end=XXRR -> True, 2 nước
   RXXR
   XRXR   (R ở 0 -> 1)
   XXRR   (R ở 1 -> 2)


True

Ví dụ thứ hai (`RXXR → XXRR`): `R` cuối cùng đã ở đích; `R` đầu tiên đi hai bước. Thứ tự xử lý **từ phải sang trái** đảm bảo xe phía trước đã ở vị trí cuối (không còn cản đường) trước khi xe phía sau tiến.

## 9. Code nộp LeetCode

Dán đoạn dưới vào LeetCode (đây là Cách 2: hai con trỏ).

In [22]:
class Solution:
    def canTransform(self, start: str, end: str) -> bool:
        n = len(start)
        i = j = 0
        while True:
            # bỏ qua các ô trống 'X' ở cả hai chuỗi
            while i < n and start[i] == 'X':
                i += 1
            while j < n and end[j] == 'X':
                j += 1
            if i >= n and j >= n:                  # đã ghép hết các xe
                return True
            if i >= n or j >= n or start[i] != end[j]:
                return False                       # số xe/loại xe khác nhau
            if start[i] == 'L' and i < j:          # L chỉ đi sang trái
                return False
            if start[i] == 'R' and i > j:          # R chỉ đi sang phải
                return False
            i, j = i + 1, j + 1


for start, end, expected in examples:
    assert Solution().canTransform(start, end) == expected
print("Solution chạy đúng 6 ví dụ")

Solution chạy đúng 6 ví dụ


## 10. Vì sao thuật toán đúng

Gọi `a_1 < a_2 < ... < a_m` là các vị trí không phải `X` của `start`, `b_1 < ... < b_m'` của `end`. Ta chứng minh: **`end` đạt được từ `start` ⇔ (i) `m = m'`, ký tự thứ `k` ở hai chuỗi giống nhau với mọi `k`, và (ii) với mọi `k`: ký tự `R` thì `a_k ≤ b_k`, ký tự `L` thì `a_k ≥ b_k`.**

### Điều kiện cần

Một nước đi chỉ hoán đổi một `L`/`R` với một `X` kề nó: **không thay đổi thứ tự** các ký tự khác `X` → (i) bất biến. Nước `RX → XR` tăng vị trí của một `R` thêm 1 và không đổi các ký tự khác; nước `XL → LX` giảm vị trí của một `L` thêm 1. Vị trí của `R` chỉ tăng, của `L` chỉ giảm → (ii) cần với mọi chuỗi đạt được.

### Điều kiện đủ

Giả sử (i) và (ii) đúng, và đặt `D = Σ_k |a_k − b_k|` (tổng quãng đường còn lại). Quy nạp theo `D`:

- `D = 0`: các vị trí trùng nhau, hai chuỗi bằng nhau.
- `D > 0`: có `k` với `a_k ≠ b_k`. Theo (ii), hoặc đó là `R` với `a_k < b_k` (cần đi phải), hoặc là `L` với `a_k > b_k` (cần đi trái).

  - *Trường hợp có `R` cần đi phải.* Chọn `R` như vậy với **chỉ số `k` lớn nhất**. Ô `a_k + 1` nằm trong chuỗi (vì `a_k + 1 ≤ b_k`). Ta khẳng định ô đó là `X`. Nếu không, ký tự ở `a_k + 1` là xe thứ `k + 1`, `a_{k+1} = a_k + 1`. Nếu xe `k + 1` là `L`: (ii) cho `b_{k+1} ≤ a_{k+1} = a_k + 1`, mà `b_k < b_{k+1}`, nên `b_k ≤ a_k`, mâu thuẫn với `a_k < b_k`. Nếu xe `k + 1` là `R`: `b_{k+1} ≥ a_{k+1}` và `b_{k+1} > b_k > a_k`, nên `b_{k+1} > a_k + 1 = a_{k+1}`, tức xe `k + 1` cũng là `R` cần đi phải với chỉ số lớn hơn `k` — mâu thuẫn với cách chọn `k`. Vậy ô `a_k + 1` là `X`; thực hiện nước `RX → XR`, `D` giảm 1 và (i), (ii) vẫn đúng (vị trí `a_k` tăng 1 vẫn `≤ b_k`, thứ tự không đổi).
  - *Trường hợp không có `R` nào cần đi phải nhưng có `L` cần đi trái.* Đối xứng: chọn `L` cần đi trái với chỉ số **nhỏ nhất**, chứng minh tương tự ô `a_k − 1` là `X`, thực hiện `XL → LX`.
  - Không còn xe nào cần di chuyển nghĩa là `D = 0` (vì (ii) cấm `R` lùi và `L` tiến).

Quy nạp xong: tồn tại dãy nước đi. ∎

**Hai con trỏ (Cách 2)** kiểm tra (i) và (ii) đồng thời: mỗi bước ghép xe thứ `k` của hai chuỗi, so `start[i] != end[j]` cho (i) (và độ dài bằng nhau nhờ `i >= n`, `j >= n` cùng lúc), so `i` với `j` cho (ii). **Cách 3** kiểm tra y hệt bằng danh sách. **Cách 4** diễn đạt (ii) bằng tiền tố: `a_k ≤ b_k` với mọi `R` ⇔ với mọi `p`, số `R` ở `start[:p]` ≥ số `R` ở `end[:p]` (vì `R` thứ `k` ở `start` xuất hiện không muộn hơn); tương tự cho `L`. **Cách 5** chính là phép quy nạp ở trên (thứ tự xử lý: `R` từ phải sang trái rồi `L` từ trái sang phải đảm bảo mọi ô cần dùng đều trống). **Cách 1** thử mọi khả năng nên đúng theo định nghĩa.

## 11. Phân tích Time / Space Complexity

Ký hiệu: `n` là độ dài chuỗi, `D` là tổng quãng đường các xe phải đi.

| Cách | Time | Space |
|---|---|---|
| 1. BFS mọi chuỗi đạt được | hàm mũ theo `n` | hàm mũ theo `n` |
| 2. Hai con trỏ | `O(n)` | `O(1)` |
| 3. Danh sách (ký tự, vị trí) | `O(n)` | `O(n)` |
| 4. Đếm tiền tố | `O(n)` | `O(n)` (do `replace`) |
| 5. Mô phỏng tham lam | `O(n + D)` ≤ `O(n²)` | `O(n)` |

Với `n = 10^4`, các cách `O(n)` chạy trong vài mili-giây. Đo thực nghiệm:

In [23]:
rng = random.Random(777)
n = 10**4
cases_big = {}
# chuỗi rất nhiều xe (không có X) trùng nhau -> cả mảng đều phải so sánh
cases_big["không X, trùng"] = ("RL" * (n // 2), "RL" * (n // 2))
# nhiều X
cases_big["nhiều X"] = ("R" + "X" * (n - 2) + "L", "X" * (n // 2 - 1) + "RL" + "X" * (n // 2 - 1))
# ngẫu nhiên
s = "".join(rng.choice("LRX") for _ in range(n))
cases_big["ngẫu nhiên (start = end)"] = (s, s)
for label, (a, b) in cases_big.items():
    out = []
    for name, cls in [("C2", Sol2), ("C3", Sol3), ("C4", Sol4), ("Solution", Solution)]:
        t0 = time.perf_counter()
        r = cls().canTransform(a, b)
        out.append(f"{name}={1000 * (time.perf_counter() - t0):6.2f}ms")
    print(f"{label:<26} -> {r} | " + " ".join(out))

không X, trùng             -> True | C2=  2.08ms C3=  3.78ms C4=  0.96ms Solution=  2.70ms
nhiều X                    -> True | C2=  0.79ms C3=  0.65ms C4=  1.55ms Solution=  1.31ms
ngẫu nhiên (start = end)   -> True | C2=  2.05ms C3=  2.43ms C4=  1.42ms Solution=  1.93ms


## 12. Bộ test

### 12.1. Ví dụ đề và edge case

**Oracle độc lập:** BFS vét cạn mọi chuỗi đạt được (Cách 1) — không dùng bất biến, hai con trỏ hay đếm. Chỉ chạy được với chuỗi ngắn, nên các test vét cạn/ngẫu nhiên đối chiếu dùng độ dài ≤ 10. Ta gom các cách vào dict `solvers`.

In [24]:
solvers = {
    "bfs": can_transform_bfs,
    "two_pointers": lambda a, b: Sol2().canTransform(a, b),
    "pairs": lambda a, b: Sol3().canTransform(a, b),
    "prefix": lambda a, b: Sol4().canTransform(a, b),
    "greedy": lambda a, b: Sol5().canTransform(a, b),
    "leetcode": lambda a, b: Solution().canTransform(a, b),
}
for start, end, expected in examples:
    for name, fn in solvers.items():
        assert fn(start, end) == expected, (name, start, end)
print("Ví dụ đề: OK cho cả 6 solver (gồm oracle BFS)")

Ví dụ đề: OK cho cả 6 solver (gồm oracle BFS)


In [25]:
edge_cases = [
    ("X", "X", True, "một ô trống"),
    ("L", "L", True, "một xe L"),
    ("R", "R", True, "một xe R"),
    ("L", "R", False, "khác loại"),
    ("XL", "LX", True, "L sang trái"),
    ("LX", "XL", False, "L không sang phải"),
    ("RX", "XR", True, "R sang phải"),
    ("XR", "RX", False, "R không sang trái"),
    ("RL", "LR", False, "R và L không đổi chỗ"),
    ("LR", "LR", True, "giữ nguyên"),
    ("XXXX", "XXXX", True, "toàn X"),
    ("XXXL", "LXXX", True, "L đi xa"),
    ("RXXX", "XXXR", True, "R đi xa"),
    ("RXXL", "XRLX", True, "ví dụ 6"),
    ("RXXL", "XXRL", True, "R đi hai bước, L đứng yên"),
    ("RXL", "XRL", True, "R sang phải một bước, chạm L"),
    ("RXL", "RLX", True, "L sang trái một bước, chạm R"),
    ("RXL", "XLR", False, "đổi thứ tự R và L"),
    ("RRX", "XRR", True, "hai R nối đuôi, phải đi từ phải sang trái"),
    ("XLL", "LLX", True, "hai L nối đuôi"),
    ("LXR", "LXR", True, "L và R tách xa không ảnh hưởng"),
    ("LXR", "XLR", False, "L sang phải không được"),
    ("RXXLRXRXL", "XRLXXRRLX", True, "ví dụ 1"),
    ("XRLXXRRLX", "RXXLRXRXL", False, "đảo ví dụ 1 (không đảo ngược được)"),
    ("RXR", "RRX", False, "R thứ hai phải lùi trái"),
    ("RXR", "XRR", True, "R đầu tiên sang phải, sát R thứ hai"),
]
for start, end, expected, note in edge_cases:
    for name, fn in solvers.items():
        assert fn(start, end) == expected, (name, note, start, end)
print(f"{len(edge_cases)} edge case: OK")

26 edge case: OK


### 12.2. Phản ví dụ cho các ý tưởng sai

- **Chỉ so số lượng `L`, `R`, `X`** (bỏ qua thứ tự/vị trí): sai với `RL → LR` (cùng số lượng nhưng không thể).
- **Chỉ so dãy xe sau khi bỏ `X`** (bỏ qua hướng đi): sai với `LX → XL` (cùng dãy xe `L`, nhưng `L` không đi sang phải được).
- **Cho `L` và `R` đi cùng chiều** (so sánh ngược): sai ở mọi trường hợp hướng.

In [26]:
def wrong_counts(start, end):
    return all(start.count(c) == end.count(c) for c in "LRX")

def wrong_no_direction(start, end):
    return start.replace("X", "") == end.replace("X", "")

def wrong_reversed(start, end):
    a = [(c, i) for i, c in enumerate(start) if c != "X"]
    b = [(c, j) for j, c in enumerate(end) if c != "X"]
    if len(a) != len(b):
        return False
    return all(c1 == c2 and ((i >= j) if c1 == "R" else (i <= j)) for (c1, i), (c2, j) in zip(a, b))

cases = [("RL", "LR"), ("LX", "XL"), ("XL", "LX"), ("RX", "XR")]
for a, b in cases:
    print(f"{a!r}->{b!r} | đúng={can_transform_bfs(a, b)!s:<5} | chỉ đếm={wrong_counts(a, b)!s:<5} | không hướng={wrong_no_direction(a, b)!s:<5} | ngược chiều={wrong_reversed(a, b)!s:<5}")
assert wrong_counts("RL", "LR") != can_transform_bfs("RL", "LR")
assert wrong_no_direction("LX", "XL") != can_transform_bfs("LX", "XL")
assert wrong_reversed("XL", "LX") != can_transform_bfs("XL", "LX")

'RL'->'LR' | đúng=False | chỉ đếm=True  | không hướng=False | ngược chiều=False
'LX'->'XL' | đúng=False | chỉ đếm=True  | không hướng=True  | ngược chiều=True 
'XL'->'LX' | đúng=True  | chỉ đếm=True  | không hướng=True  | ngược chiều=False
'RX'->'XR' | đúng=True  | chỉ đếm=True  | không hướng=True  | ngược chiều=False


### 12.3. Đối chiếu mọi cách bằng vét cạn toàn bộ input nhỏ

Duyệt **mọi cặp** `(start, end)` trên bảng chữ cái `{L, R, X}` với độ dài `1..5` (tổng `Σ 3^n · 3^n` cặp — với `n = 5` là `59 049` cặp) và so 6 solver (gồm BFS oracle, tính tập đạt được **một lần** cho mỗi `start`).

In [27]:
from itertools import product

checked = 0
positives = 0
for n in range(1, 6):
    strings = ["".join(p) for p in product("LRX", repeat=n)]
    for start in strings:
        reach = reachable(start)
        for end in strings:
            expected = end in reach
            positives += expected
            for name in ("two_pointers", "pairs", "prefix", "greedy", "leetcode"):
                assert solvers[name](start, end) == expected, (name, start, end)
            checked += 1
print("đã kiểm", checked, "cặp (start, end), trong đó", positives, "cặp True: tất cả khớp BFS")

đã kiểm 66429 cặp (start, end), trong đó 937 cặp True: tất cả khớp BFS


### 12.4. Test ngẫu nhiên có seed (đúng ràng buộc) và test cặp "chắc chắn True"

Hai nhóm: (a) cặp ngẫu nhiên độ dài `6..10` đối chiếu BFS; (b) cặp **chắc chắn True** tạo bằng cách áp dụng ngẫu nhiên nhiều nước đi hợp lệ lên `start` (độ dài tới `10^4`, không dùng BFS), và cặp "True rồi bị phá" (đổi một ký tự) — cho kết quả mong đợi tính bằng hai cách độc lập (`two_pointers` so với `pairs`/`prefix`/`greedy`).

In [28]:
rng = random.Random(7770)

def random_moves(start, rng, steps):
    s = list(start)
    n = len(s)
    for _ in range(steps):
        i = rng.randrange(n - 1) if n > 1 else 0
        if n > 1 and s[i] == "R" and s[i + 1] == "X":
            s[i], s[i + 1] = "X", "R"
        elif n > 1 and s[i] == "X" and s[i + 1] == "L":
            s[i], s[i + 1] = "L", "X"
    return "".join(s)

# (a) ngẫu nhiên nhỏ, đối chiếu BFS
trues = 0
for _ in range(3000):
    n = rng.randint(6, 10)
    start = "".join(rng.choice("LRXX") for _ in range(n))
    end = random_moves(start, rng, 40) if rng.random() < 0.5 else "".join(rng.choice("LRXX") for _ in range(n))
    expected = can_transform_bfs(start, end)
    trues += expected
    for name in ("two_pointers", "pairs", "prefix", "greedy", "leetcode"):
        assert solvers[name](start, end) == expected, (name, start, end)
print("3000 cặp ngẫu nhiên nhỏ khớp BFS;", trues, "cặp True")

# (b) lớn: True được xây dựng, rồi phá hoại
for _ in range(40):
    n = rng.choice([100, 1000, 10**4])
    start = "".join(rng.choice("LRXXX") for _ in range(n))
    end = random_moves(start, rng, 5 * n)
    for name in ("two_pointers", "pairs", "prefix", "leetcode"):
        assert solvers[name](start, end) is True, (name, n)
    # phá: đổi một ký tự khác X thành loại còn lại -> dãy xe khác nhau -> False
    idx = [i for i, c in enumerate(end) if c != "X"]
    if idx:
        k = rng.choice(idx)
        bad = end[:k] + ("L" if end[k] == "R" else "R") + end[k + 1:]
        for name in ("two_pointers", "pairs", "prefix", "leetcode"):
            assert solvers[name](start, bad) is False, (name, n)
print("40 cặp lớn: True xây dựng được và False khi phá hoại: OK")

3000 cặp ngẫu nhiên nhỏ khớp BFS; 1462 cặp True


40 cặp lớn: True xây dựng được và False khi phá hoại: OK


### 12.5. Tính chất bổ sung và trường hợp tệ nhất

Các tính chất: (a) phản xạ `canTransform(s, s)` luôn `True`; (b) **bắc cầu**: nếu `a → b` và `b → c` thì `a → c`; (c) **không đối xứng**: nếu `a → b`, `a ≠ b` thì `b ↛ a`; (d) đáp án `True` kéo theo cùng số `L` và cùng số `R`.

In [29]:
rng = random.Random(77700)
for _ in range(1500):
    n = rng.randint(1, 9)
    a = "".join(rng.choice("LRXX") for _ in range(n))
    b = random_moves(a, rng, 20)
    c = random_moves(b, rng, 20)
    assert Solution().canTransform(a, a)                                    # (a)
    assert Solution().canTransform(a, b) and Solution().canTransform(b, c)
    assert Solution().canTransform(a, c)                                    # (b)
    if a != b:
        assert not Solution().canTransform(b, a)                            # (c)
    assert a.count("L") == b.count("L") and a.count("R") == b.count("R")    # (d)
print("tính chất phản xạ, bắc cầu, không đối xứng: OK")

n = 10**4
worst = {
    "toàn X": ("X" * n, "X" * n),
    "một R đi hết chuỗi": ("R" + "X" * (n - 1), "X" * (n - 1) + "R"),
    "một L đi hết chuỗi": ("X" * (n - 1) + "L", "L" + "X" * (n - 1)),
    "R tuần hoàn": ("RX" * (n // 2), "XR" * (n // 2)),
    "L tuần hoàn": ("XL" * (n // 2), "LX" * (n // 2)),
    "độ dài khác loại": ("L" * n, "R" * n),
}
expected = {"toàn X": True, "một R đi hết chuỗi": True, "một L đi hết chuỗi": True,
            "R tuần hoàn": True, "L tuần hoàn": True, "độ dài khác loại": False}
for label, (a, b) in worst.items():
    for name in ("two_pointers", "pairs", "prefix", "leetcode"):
        assert solvers[name](a, b) is expected[label], (name, label)
print("trường hợp tệ nhất n = 10^4: OK")

tính chất phản xạ, bắc cầu, không đối xứng: OK
trường hợp tệ nhất n = 10^4: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Chỉ so số lượng `L`, `R`, `X`

**Code sai:**

In [30]:
class WrongSolution1:
    def canTransform(self, start, end):
        return sorted(start) == sorted(end)          # cùng số lượng L, R, X

for a, b in [("RL", "LR"), ("RXXL", "XRLX"), ("LX", "XL")]:
    print(f"{a}->{b} | sai: {WrongSolution1().canTransform(a, b)} | đúng: {Solution().canTransform(a, b)}")

RL->LR | sai: True | đúng: False
RXXL->XRLX | sai: True | đúng: True
LX->XL | sai: True | đúng: False


**Vì sao sai:** số lượng giống nhau không đủ — **thứ tự** `L`/`R` bất biến (`RL` khác `LR`) và **hướng** đi cũng bị ràng buộc (`LX → XL` sai dù cùng số lượng). **Code đúng:** so dãy sau khi bỏ `X` và kiểm tra vị trí theo hướng.

### Lỗi 2: Chỉ so dãy sau khi bỏ `X` (quên điều kiện hướng)

In [31]:
class WrongSolution2:
    def canTransform(self, start, end):
        return start.replace("X", "") == end.replace("X", "")

for a, b in [("LX", "XL"), ("XR", "RX"), ("RXXL", "XRLX")]:
    print(f"{a}->{b} | sai: {WrongSolution2().canTransform(a, b)} | đúng: {Solution().canTransform(a, b)}")

LX->XL | sai: True | đúng: False
XR->RX | sai: True | đúng: False
RXXL->XRLX | sai: True | đúng: True


**Vì sao sai:** `L` chỉ đi **trái**, `R` chỉ đi **phải**. `LX → XL` cần `L` đi phải nên **không** thực hiện được dù dãy xe giống nhau. **Code đúng:** thêm kiểm tra `start[i] == 'L' and i < j` và `start[i] == 'R' and i > j`.

### Lỗi 3: So sánh ngược chiều (nhầm `<` và `>`)

In [32]:
class WrongSolution3:
    def canTransform(self, start, end):
        n = len(start)
        i = j = 0
        while True:
            while i < n and start[i] == "X":
                i += 1
            while j < n and end[j] == "X":
                j += 1
            if i >= n and j >= n:
                return True
            if i >= n or j >= n or start[i] != end[j]:
                return False
            if start[i] == "L" and i > j:            # sai chiều
                return False
            if start[i] == "R" and i < j:            # sai chiều
                return False
            i, j = i + 1, j + 1

for a, b in [("XL", "LX"), ("RX", "XR"), ("LX", "XL")]:
    print(f"{a}->{b} | sai: {WrongSolution3().canTransform(a, b)} | đúng: {Solution().canTransform(a, b)}")

XL->LX | sai: False | đúng: True
RX->XR | sai: False | đúng: True
LX->XL | sai: True | đúng: False


**Vì sao sai:** `L` ở `start` vị trí `i`, đích `j`: `L` đi sang **trái** nên cần `j ≤ i`; vi phạm khi `i < j`. Viết ngược dấu thì đảo hết kết quả. **Mẹo:** tự kiểm bằng `XL → LX` (phải `True`) và `LX → XL` (phải `False`).

### Lỗi 4: Quên bỏ qua `X` ở một trong hai con trỏ (hoặc `X` làm lệch cặp)

In [33]:
class WrongSolution4:
    def canTransform(self, start, end):
        n = len(start)
        i = 0
        j = 0
        while i < n and j < n:
            if start[i] == "X":
                i += 1
                continue
            if end[j] == "X":
                j += 1
                continue
            if start[i] != end[j]:
                return False
            if start[i] == "L" and i < j:
                return False
            if start[i] == "R" and i > j:
                return False
            i += 1
            j += 1
        return True                                    # sai: một bên còn xe chưa ghép

for a, b in [("RXX", "XXX"), ("XXX", "XXL"), ("XL", "XX")]:
    print(f"{a}->{b} | sai: {WrongSolution4().canTransform(a, b)} | đúng: {Solution().canTransform(a, b)}")

RXX->XXX | sai: True | đúng: False
XXX->XXL | sai: True | đúng: False
XL->XX | sai: True | đúng: False


**Vì sao sai:** vòng lặp thoát khi **một** bên hết, bỏ qua trường hợp bên kia còn xe chưa ghép (số xe khác nhau). Phải kiểm tra **cả hai** cùng hết: `i >= n and j >= n`, và nếu chỉ một bên hết thì `False`.

### Lỗi 5: Thử mô phỏng bằng `replace` lặp lại cho đến khi không đổi

In [34]:
def wrong_replace_loop(start, end):
    s = start
    for _ in range(len(start) + 1):
        s = s.replace("XL", "LX").replace("RX", "XR")     # áp mọi nước tới tối đa
        if s == end:
            return True
    return s == end

for a, b in [("RXXL", "XRLX"), ("RXXL", "XXRL"), ("RXXRX", "XXRXR")]:
    print(f"{a}->{b} | sai: {wrong_replace_loop(a, b)} | đúng: {Solution().canTransform(a, b)}")

RXXL->XRLX | sai: True | đúng: True
RXXL->XXRL | sai: False | đúng: True
RXXRX->XXRXR | sai: True | đúng: True


**Vì sao sai:** áp dụng **mọi** nước đi cùng lúc ép các xe đi **hết tốc độ**, nhưng đích có thể yêu cầu một xe **dừng giữa chừng** (như `RXXL → XXRL`: `R` đi hai bước rồi dừng). Mô phỏng "đủ mạnh" phải biết **đích** để quyết định dừng — chính là ý tưởng của Cách 5.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy **"đổi chỗ kề nhau theo quy tắc cố định, hỏi có đạt được không"** → tìm **bất biến** (thứ gì không bao giờ đổi) thay vì mô phỏng.
- Ký tự **không thể vượt nhau** ⇒ thứ tự tương đối là bất biến ⇒ so sánh **dãy sau khi bỏ ký tự "trống"**.
- Ký tự chỉ đi **một chiều** ⇒ so sánh vị trí theo hướng (`R`: `a ≤ b`; `L`: `a ≥ b`).
- Hai chuỗi, so từng phần tử "không phải trống" ⇒ **hai con trỏ** cùng bỏ qua phần tử trống.
- Dạng bài này còn xuất hiện ở LeetCode **2337. Move Pieces to Obtain a String** (dùng `_` thay `X`, cách làm y hệt).
- Nếu chưa chắc điều kiện, **dùng BFS vét cạn** trên chuỗi ngắn làm oracle để kiểm chứng giả thuyết (đúng như ta làm ở mục 12).
- Luôn thử: `XL → LX` (True), `LX → XL` (False), `RL → LR` (False), toàn `X`, độ dài 1.

## 15. Biến thể

### 15.1. Số nước đi tối thiểu

Nếu có thể biến đổi, **số nước tối thiểu** chính là tổng quãng đường `D = Σ|a_k − b_k|` (mỗi nước dịch đúng một xe đúng một ô và không bao giờ phải lùi). Nếu không thể, trả `-1`. Ta kiểm chứng bằng BFS tính khoảng cách.

In [35]:
def min_moves(start, end):
    if not Solution().canTransform(start, end):
        return -1
    a = [i for i, c in enumerate(start) if c != "X"]
    b = [j for j, c in enumerate(end) if c != "X"]
    return sum(abs(x - y) for x, y in zip(a, b))

def bfs_distance(start, end):
    if start == end:
        return 0
    seen = {start}
    queue = deque([(start, 0)])
    while queue:
        s, d = queue.popleft()
        for i in range(len(s) - 1):
            pair = s[i:i + 2]
            if pair == "XL":
                t = s[:i] + "LX" + s[i + 2:]
            elif pair == "RX":
                t = s[:i] + "XR" + s[i + 2:]
            else:
                continue
            if t == end:
                return d + 1
            if t not in seen:
                seen.add(t)
                queue.append((t, d + 1))
    return -1

print(min_moves("RXXLRXRXL", "XRLXXRRLX"), min_moves("LX", "XL"), min_moves("RXXX", "XXXR"))
rng = random.Random(151)
for _ in range(1500):
    n = rng.randint(1, 9)
    a = "".join(rng.choice("LRXX") for _ in range(n))
    b = random_moves(a, rng, 15) if rng.random() < 0.6 else "".join(rng.choice("LRXX") for _ in range(n))
    assert min_moves(a, b) == bfs_distance(a, b), (a, b)
print("min_moves khớp BFS khoảng cách ngắn nhất")

4 -1 3
min_moves khớp BFS khoảng cách ngắn nhất


### 15.2. Dãy nước đi tường minh (Cách 5) và kiểm chứng

`construct_moves` ở mục 7.4 trả về một dãy nước đi; ta **phát lại** từng nước trên `start` bằng `apply_moves` (mỗi nước đều kiểm tra hợp lệ) và so với `end`; độ dài dãy đúng bằng `min_moves`.

In [36]:
rng = random.Random(152)
for _ in range(1500):
    n = rng.randint(1, 12)
    a = "".join(rng.choice("LRXX") for _ in range(n))
    b = random_moves(a, rng, 25)
    ok, moves = construct_moves(a, b)
    assert ok
    assert apply_moves(a, moves) == b
    assert len(moves) == min_moves(a, b)
print("dãy nước đi dựng được luôn hợp lệ, đến đúng đích và đạt số nước tối thiểu")

dãy nước đi dựng được luôn hợp lệ, đến đúng đích và đạt số nước tối thiểu


### 15.3. Cho phép đi cả hai chiều (`XL ↔ LX`, `RX ↔ XR`)

Nếu mỗi nước đi là **hai chiều** (`XL ↔ LX` và `RX ↔ XR`), `L` và `R` có thể đi cả hai hướng nhưng vẫn **không vượt qua nhau**. Khi đó điều kiện hướng biến mất; chỉ cần **dãy sau khi bỏ `X` bằng nhau**. Ta kiểm chứng bằng BFS hai chiều.

In [37]:
def reachable_bi(start):
    seen = {start}
    queue = deque([start])
    while queue:
        s = queue.popleft()
        for i in range(len(s) - 1):
            pair = s[i:i + 2]
            if pair == "XL":
                t = s[:i] + "LX" + s[i + 2:]
            elif pair == "LX":
                t = s[:i] + "XL" + s[i + 2:]
            elif pair == "RX":
                t = s[:i] + "XR" + s[i + 2:]
            elif pair == "XR":
                t = s[:i] + "RX" + s[i + 2:]
            else:
                continue
            if t not in seen:
                seen.add(t)
                queue.append(t)
    return seen

def can_transform_bidirectional(start, end):
    return start.replace("X", "") == end.replace("X", "")

print(can_transform_bidirectional("LX", "XL"), can_transform_bidirectional("RL", "LR"))
for n in range(1, 6):
    strings = ["".join(p) for p in product("LRX", repeat=n)]
    for start in strings:
        reach = reachable_bi(start)
        for end in strings:
            assert (end in reach) == can_transform_bidirectional(start, end), (start, end)
print("biến thể hai chiều: chỉ cần so dãy sau khi bỏ X (kiểm chứng vét cạn n <= 5)")

True False
biến thể hai chiều: chỉ cần so dãy sau khi bỏ X (kiểm chứng vét cạn n <= 5)


### 15.4. Các bài liên quan trên LeetCode (đọc thêm)

- **2337. Move Pieces to Obtain a String:** cùng bài với `_` thay `X`, các quân `L`, `R`.
- **844. Backspace String Compare, 392. Is Subsequence:** hai con trỏ trên hai chuỗi với "bỏ qua" ký tự.
- **763. Partition Labels, 1047. Remove All Adjacent Duplicates:** xử lý chuỗi bằng bất biến/ngăn xếp.
- **1052/1160...:** các bài "tìm bất biến" trên chuỗi/đồ thị trạng thái.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. BFS mọi chuỗi đạt được | liệt kê đồ thị trạng thái | hàm mũ | hàm mũ | chỉ để hiểu đề / oracle với `n ≤ 10` |
| 2. Hai con trỏ | bỏ `X`, so xe thứ `k` và hướng | `O(n)` | `O(1)` | **cách nộp bài chuẩn** |
| 3. Danh sách (ký tự, vị trí) | cùng ý, viết rõ ràng bằng `zip` | `O(n)` | `O(n)` | muốn code dễ đọc, ít lỗi chỉ số |
| 4. Đếm tiền tố | so số `R`, `L` trong từng tiền tố | `O(n)` | `O(n)` | cách nhìn "dòng chảy" của các xe |
| 5. Mô phỏng tham lam | thực sự dịch từng xe tới đích | `O(n + D)` | `O(n)` | cần **dãy nước đi** hoặc số nước tối thiểu |

## 17. 5 bài tập vận dụng (dễ → khó)

Cell **TODO** chỉ gán `None` hoặc định nghĩa hàm rỗng, **không được gọi** khi chạy notebook. Đáp án ở mục 19.

**Bài 1 (đoán output).** Dự đoán rồi chạy:

In [38]:
print(Solution().canTransform("XXRXL", "XRXXL"))
print(Solution().canTransform("XRXXL", "XXRXL"))
print(Solution().canTransform("LXXR", "LXXR"))
print(Solution().canTransform("RXXRX", "XXRXR"))

False
True
True
True


**Bài 2 (tính tay).** Với `start = "RXXLRXRXL"` và `end = "XRLXXRRLX"`: viết **danh sách vị trí các ký tự khác `X`** của `start` và của `end` (theo thứ tự). Gán tuple `(vi_tri_start, vi_tri_end)` vào `ans2` (đang là `None`), mỗi phần tử là list các số nguyên.

In [39]:
ans2 = None      # TODO: tuple (list vị trí start, list vị trí end)

**Bài 3 (điền TODO).** Hoàn thành `can_transform_pairs(start, end)` theo **Cách 3** (danh sách `(ký tự, vị trí)` rồi so từng cặp).

In [40]:
def can_transform_pairs(start, end):
    a = None      # TODO: list (c, i) với c != "X"
    b = None      # TODO: list (c, j) với c != "X"
    # TODO: nếu độ dài khác nhau trả False
    # TODO: với từng cặp ((c1, i), (c2, j)): c1 == c2; R cần i <= j; L cần i >= j
    return None

**Bài 4 (tự viết lại).** Không nhìn mục 7, tự viết `my_can_transform(start, end)` theo **hai con trỏ** (Cách 2).

In [41]:
def my_can_transform(start, end):
    # TODO
    pass

**Bài 5 (mini challenge).** (a) `moves_needed(start, end)`: số nước đi **tối thiểu** (hoặc `-1` nếu không thể). (b) `can_transform_both(start, end)`: bài biến thể cho phép **cả hai chiều** (`XL ↔ LX`, `RX ↔ XR`). Ví dụ `moves_needed("RXXX", "XXXR") = 3`; `can_transform_both("LX", "XL") = True`.

In [42]:
def moves_needed(start, end):
    # TODO
    pass

def can_transform_both(start, end):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):**
- Bài 1: xem hướng của mỗi xe: `R` chỉ sang phải, `L` chỉ sang trái.
- Bài 2: đánh số vị trí từ `0` và bỏ qua `X`.
- Bài 3: dùng `enumerate` và danh sách comprehension như mục 3.3.
- Bài 4: hai con trỏ `i`, `j`; vòng ngoài `while True`.
- Bài 5: số nước tối thiểu liên quan tới tổng khoảng cách giữa vị trí tương ứng.

**Mức 2 (vừa):**
- Bài 1: `XXRXL → XRXXL`: `R` ở vị trí 2 phải về vị trí 1 (lùi) → sai.
- Bài 2: `start`: 0, 3, 4, 6, 8; `end`: 1, 2, 5, 6, 7.
- Bài 3: `if c1 == "R" and i > j: return False`, `if c1 == "L" and i < j: return False`.
- Bài 4: sau khi `i`, `j` đã bỏ `X`, xét ba trường hợp: cả hai hết → `True`; một bên hết hoặc khác loại → `False`; vi phạm hướng → `False`.
- Bài 5(a): `sum(abs(a_k - b_k))` nếu `can_transform`; (b) chỉ cần `start.replace("X", "") == end.replace("X", "")`.

**Mức 3 (gần đáp án):**
- Bài 3: xem Cách 3 ở mục 7.2.
- Bài 4: xem Cách 2 ở mục 7.1.
- Bài 5: xem 15.1 (số nước tối thiểu) và 15.3 (hai chiều).

## 19. Đáp án tham khảo

In [43]:
# Bài 1: False, True, True, True
assert Solution().canTransform("XXRXL", "XRXXL") is False
assert Solution().canTransform("XRXXL", "XXRXL") is True       # R đi từ 1 sang 2 (phải), L đứng yên
assert Solution().canTransform("LXXR", "LXXR") is True
assert Solution().canTransform("RXXRX", "XXRXR") is True
print("Bài 1 OK: False, True, True, True")

# Bài 2
ans2_ref = ([0, 3, 4, 6, 8], [1, 2, 5, 6, 7])
assert ans2_ref == (
    [i for i, c in enumerate("RXXLRXRXL") if c != "X"],
    [i for i, c in enumerate("XRLXXRRLX") if c != "X"],
)
print("Bài 2 OK:", ans2_ref)

Bài 1 OK: False, True, True, True
Bài 2 OK: ([0, 3, 4, 6, 8], [1, 2, 5, 6, 7])


In [44]:
# Bài 3
def can_transform_pairs_ref(start, end):
    a = [(c, i) for i, c in enumerate(start) if c != "X"]
    b = [(c, j) for j, c in enumerate(end) if c != "X"]
    if len(a) != len(b):
        return False
    for (c1, i), (c2, j) in zip(a, b):
        if c1 != c2:
            return False
        if c1 == "R" and i > j:
            return False
        if c1 == "L" and i < j:
            return False
    return True

# Bài 4
def my_can_transform_ref(start, end):
    n = len(start)
    i = j = 0
    while True:
        while i < n and start[i] == "X":
            i += 1
        while j < n and end[j] == "X":
            j += 1
        if i >= n and j >= n:
            return True
        if i >= n or j >= n or start[i] != end[j]:
            return False
        if start[i] == "L" and i < j:
            return False
        if start[i] == "R" and i > j:
            return False
        i, j = i + 1, j + 1

rng = random.Random(1903)
for _ in range(3000):
    n = rng.randint(1, 9)
    a = "".join(rng.choice("LRXX") for _ in range(n))
    b = random_moves(a, rng, 20) if rng.random() < 0.5 else "".join(rng.choice("LRXX") for _ in range(n))
    expected = can_transform_bfs(a, b)
    assert can_transform_pairs_ref(a, b) == expected
    assert my_can_transform_ref(a, b) == expected
print("Bài 3, 4 OK")

Bài 3, 4 OK


In [45]:
# Bài 5
def moves_needed_ref(start, end):
    if not my_can_transform_ref(start, end):
        return -1
    a = [i for i, c in enumerate(start) if c != "X"]
    b = [j for j, c in enumerate(end) if c != "X"]
    return sum(abs(x - y) for x, y in zip(a, b))

def can_transform_both_ref(start, end):
    return start.replace("X", "") == end.replace("X", "")

assert moves_needed_ref("RXXX", "XXXR") == 3
assert moves_needed_ref("LX", "XL") == -1
assert can_transform_both_ref("LX", "XL") is True and can_transform_both_ref("RL", "LR") is False
rng = random.Random(1905)
for _ in range(1500):
    n = rng.randint(1, 9)
    a = "".join(rng.choice("LRXX") for _ in range(n))
    b = random_moves(a, rng, 15) if rng.random() < 0.5 else "".join(rng.choice("LRXX") for _ in range(n))
    assert moves_needed_ref(a, b) == bfs_distance(a, b), (a, b)
    assert can_transform_both_ref(a, b) == (b in reachable_bi(a)), (a, b)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template hai con trỏ (cách nộp):**

```python
class Solution:
    def canTransform(self, start, end):
        n = len(start)
        i = j = 0
        while True:
            while i < n and start[i] == 'X':
                i += 1
            while j < n and end[j] == 'X':
                j += 1
            if i >= n and j >= n:
                return True
            if i >= n or j >= n or start[i] != end[j]:
                return False
            if start[i] == 'L' and i < j:
                return False
            if start[i] == 'R' and i > j:
                return False
            i, j = i + 1, j + 1
```

**Template danh sách (ký tự, vị trí):**

```python
a = [(c, i) for i, c in enumerate(start) if c != 'X']
b = [(c, j) for j, c in enumerate(end) if c != 'X']
if len(a) != len(b): return False
for (c1, i), (c2, j) in zip(a, b):
    if c1 != c2 or (c1 == 'R' and i > j) or (c1 == 'L' and i < j):
        return False
return True
```

**Câu thần chú:** *bỏ `X`, so xe thứ `k`; `R` chỉ đi phải (`i ≤ j`), `L` chỉ đi trái (`i ≥ j`); hai xe không vượt nhau.*

**Checklist trước khi submit:**

- [ ] Đã so **dãy `L`/`R` sau khi bỏ `X`** (thứ tự bất biến), không chỉ đếm số lượng?
- [ ] Hướng đúng: `R` cần `i <= j`, `L` cần `i >= j` (thử `XL → LX` = True, `LX → XL` = False)?
- [ ] Cả hai con trỏ bỏ `X`; kiểm tra "cả hai cùng hết" mới trả `True`?
- [ ] Chỉ một bên hết hoặc khác loại → `False`?
- [ ] Đã thử: một ký tự, toàn `X`, `RL → LR`, `RRX → XRR`, `n = 10^4`?